# Cache-aside, LLM response cache, and `pgvector`

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 30 §30.4, §30.5, §30.6 · type: walkthrough*

**One-line promise:** implement cache-aside with a TTL, extend it to cache **LLM completions** keyed by (model + inputs), store and top-k query embeddings in `pgvector`, and weigh one-database simplicity against a dedicated vector store.

## 🧠 Why this matters

The fastest query is the one you never run. **Redis** — an in-memory key–value store — sits in front of slower stores and expensive computations; the default pattern is **cache-aside**: check the cache, and on a miss fetch from the source and backfill with a time-to-live (TTL). For AI systems Redis pulls double duty as the home for **LLM response caching** — an identical prompt (same model, same inputs) returns a cached completion, saving real money and latency on repeated requests (semantic caching builds on this in Ch 40). The other half of this chapter is **vector storage**: keeping embeddings *in Postgres* via `pgvector` means one database, one backup, and transactional consistency between your rows and their embeddings — often the pragmatic winner over a dedicated vector DB. The thread running through both: **cache invalidation is hard** ("how wrong, for how long?"), and every extra datastore is a permanent operational cost.

## Objectives & prereqs

**By the end you can:**
- Implement the book's `get_profile` **cache-aside** with a TTL and explain hit vs miss.
- Extend cache-aside to cache **LLM completions** keyed by `(model, inputs)` and measure tokens/$ saved.
- Insert embeddings and run a **top-k cosine** similarity query (the `pgvector` shape).
- Argue `pgvector`-in-Postgres vs a dedicated vector DB on **operational cost**, not hype.
- Apply the chapter's data-layer **operability** checklist (tested backups, replica lag, lifecycle).

**Prereqs:** notebook **30-01** (schema); Ch 13 (embeddings/RAG) for the vector half.

> ⚙️ **Runs free & offline.** Defaults to a **dict-backed fake Redis** and a **pure-Python cosine** vector search — no containers, no key. The LLM-cache cell is **mockable**: `MOCK=1` (default) returns a canned completion from `data/`; `MOCK=0` makes ~2 short live Anthropic calls to populate the cache once.

In [ ]:
# --- Setup: imports, env, and the MOCK switch ---------------------------------
# Stdlib + python-dotenv only by default. (anthropic, from requirements.txt, is
# imported lazily ONLY on the MOCK=0 live path.)
import os
import json
import time
import math
import hashlib
import pathlib

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# MOCK=1 (default): fake Redis (a dict), NumPy-free cosine search, and a CANNED LLM
#   completion read from data/ — runs free, offline, deterministically.
# MOCK=0: real Redis via REDIS_URL + ~2 short live Anthropic calls to seed the LLM cache.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

DATA = pathlib.Path("data")
print(f"MOCK mode: {MOCK}")
if not MOCK:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise SystemExit("MOCK=0 but ANTHROPIC_API_KEY is unset — add it to .env or stay in mock mode.")
    # Live LLM cache costs ~2 short completions, populated ONCE (every later hit is free).
    print("LIVE: will make up to 2 short Anthropic calls to seed the cache (~tens of tokens).")

## 1 · A fake Redis with TTL semantics

We model just enough of Redis to teach the pattern: `get`, `set(..., ex=ttl)`, and key **expiry**. The interface mirrors the `redis` client so swapping in the real thing (`MOCK=0`) is a one-line change. Everything stays in a dict — no container.

In [ ]:
class FakeRedis:
    """Dict-backed stand-in for Redis: get / set-with-TTL / delete, with expiry.
    Same method shape as redis-py, so MOCK=0 swaps in the real client unchanged."""
    def __init__(self):
        self._store = {}   # key -> (value, expires_at_or_None)

    def get(self, key):
        item = self._store.get(key)
        if item is None:
            return None
        value, expires = item
        if expires is not None and time.monotonic() > expires:
            del self._store[key]          # lazily evict on read (Redis does TTL eviction)
            return None
        return value

    def set(self, key, value, ex=None):
        expires = time.monotonic() + ex if ex else None
        self._store[key] = (value, expires)

    def delete(self, key):
        self._store.pop(key, None)         # explicit invalidation on write

if MOCK:
    redis = FakeRedis()
else:
    import redis as redis_lib            # from requirements.txt
    redis = redis_lib.from_url(os.environ["REDIS_URL"])
print("redis ready:", "FakeRedis (in-process)" if MOCK else "live Redis")

## 2 · The book's `get_profile`: cache-aside with a TTL

This is §30.4's pattern, made synchronous for the notebook. Check Redis; on a **hit** return immediately; on a **miss** fetch from the (simulated slow) database, then **backfill** the cache with a 5-minute TTL so the next read is fast.

🔮 **Predict:** we read the same profile **twice**. Which read hits the database, and which is served from cache? What's the ratio of their latencies?

In [ ]:
DB_LATENCY = 0.05  # 50 ms: a slow source query we want to avoid repeating

def db_fetch_profile(user_id):
    time.sleep(DB_LATENCY)               # stand-in for SELECT * FROM profiles WHERE id = ?
    return {"id": user_id, "name": f"User {user_id}", "plan": "pro"}

def get_profile(user_id):
    key = f"profile:{user_id}"
    cached = redis.get(key)
    if cached is not None:
        return json.loads(cached), "HIT"          # served from cache
    profile = db_fetch_profile(user_id)           # MISS -> hit the source
    redis.set(key, json.dumps(profile), ex=300)   # backfill, 5-min TTL
    return profile, "MISS"

t0 = time.perf_counter(); _, s1 = get_profile(42); t1 = time.perf_counter()
_, s2 = get_profile(42); t2 = time.perf_counter()

miss_ms = (t1 - t0) * 1000
hit_ms  = (t2 - t1) * 1000
print(f"1st read: {s1}  ({miss_ms:6.2f} ms)   <- paid the DB query")
print(f"2nd read: {s2}  ({hit_ms:6.2f} ms)   <- served from cache")
print(f"cache made the repeat read ~{miss_ms / max(hit_ms, 0.001):.0f}x faster")

**What you just saw.** The first read **missed** and paid the 50 ms DB query, then backfilled Redis. The second read **hit** and returned in microseconds. That's the whole value proposition: pay the slow source once, serve every repeat from memory until the TTL expires.

## 3 · LLM response caching: the same pattern, applied to completions

An identical prompt to an identical model is a deterministic-enough request to cache: key on `(model, inputs)`, and a repeat returns the stored completion for **zero tokens and ~zero latency**. This is the foundation Ch 40's *semantic* cache builds on (which also matches *near*-identical prompts).

In `MOCK=1` the "live call" is replaced by a canned completion from `data/canned_completion.json`, so the hit/miss + cost-saving story runs free.

In [ ]:
def llm_cache_key(model, prompt):
    # Hash (model + inputs) into a stable key. ANY change to model or prompt is a new key.
    raw = json.dumps({"model": model, "prompt": prompt}, sort_keys=True)
    return "llm:" + hashlib.sha256(raw.encode()).hexdigest()[:16]

CANNED = json.loads((DATA / "canned_completion.json").read_text(encoding="utf-8"))

def call_llm(model, prompt):
    """MOCK=1: return the canned completion (free). MOCK=0: one live Anthropic call."""
    if MOCK:
        return CANNED["completion"], CANNED["usage"]
    import anthropic                                   # from requirements.txt
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model=model, max_tokens=64,
        messages=[{"role": "user", "content": prompt}],
    )
    text = next((b.text for b in msg.content if b.type == "text"), "")
    usage = {"input_tokens": msg.usage.input_tokens, "output_tokens": msg.usage.output_tokens}
    return text, usage

def cached_completion(model, prompt):
    key = llm_cache_key(model, prompt)
    cached = redis.get(key)
    if cached is not None:
        return json.loads(cached), "HIT", {"input_tokens": 0, "output_tokens": 0}
    text, usage = call_llm(model, prompt)              # MISS -> real (or canned) call
    redis.set(key, json.dumps(text), ex=3600)          # cache the completion, 1-hour TTL
    return text, "MISS", usage

print("LLM cache ready (key = hash of model + prompt)")

🔮 **Predict:** we send the **same** (model, prompt) twice. The first call costs `input+output` tokens; what does the second cost? And what would a *different* prompt cost — hit or miss?

In [ ]:
MODEL = CANNED["model"]            # e.g. claude-sonnet-4-6
PROMPT = CANNED["prompt"]

r1, s1, u1 = cached_completion(MODEL, PROMPT)
r2, s2, u2 = cached_completion(MODEL, PROMPT)             # identical -> should hit
r3, s3, u3 = cached_completion(MODEL, PROMPT + " Be terse.")  # changed -> new key -> miss

def toks(u): return u["input_tokens"] + u["output_tokens"]

print(f"call 1: {s1}  tokens billed: {toks(u1):2d}")
print(f"call 2: {s2}  tokens billed: {toks(u2):2d}   <- free: identical request")
print(f"call 3: {s3}  tokens billed: {toks(u3):2d}   <- changed prompt = new key = miss")

# Cost framing: tokens saved across a realistic repeat-heavy workload.
REPEATS = 1000                                            # 1000 identical requests
naive = REPEATS * toks(u1)
cached = toks(u1) + (REPEATS - 1) * 0                     # pay once, then free hits
print(f"\nOver {REPEATS} identical requests: naive={naive} tokens, cached={cached} tokens "
      f"=> {100 * (1 - cached / naive):.1f}% saved")

**What you just saw.** Call 2 was a **HIT** — same model, same prompt — so it cost **zero tokens** and returned instantly. Call 3 changed the prompt, so it's a different key and a **MISS**. On a repeat-heavy workload (FAQ answers, deduped batch jobs, retried requests) the cache turns N paid calls into one. Ch 40's semantic cache widens this to *near*-identical prompts.

> ⚠️ **Pitfall — cache invalidation.** "There are only two hard things in computer science: cache invalidation and naming things." Stale cache is a top source of baffling "works for some users" bugs. Prefer **short TTLs** *plus* **explicit invalidation on write** — delete or update the key the moment the underlying data changes. And never cache what you can't afford to be briefly stale. Always ask: **how wrong can this entry be, and for how long?**

In [ ]:
# Make invalidation concrete: a profile changes, so we DELETE its key on write.
def update_profile_plan(user_id, new_plan):
    # 1) write to the source of truth, then 2) invalidate the cached copy.
    # (db UPDATE elided) ...
    redis.delete(f"profile:{user_id}")     # explicit invalidation — next read repopulates

get_profile(42)                            # warm the cache (HIT on repeat)
update_profile_plan(42, "enterprise")      # plan changed -> key deleted
_, status = get_profile(42)                # forced to re-read the source of truth
print(f"after write+invalidate, next read is a {status} (re-fetches fresh data)")
print("Short TTL is the safety net; explicit delete-on-write is the precise tool.")

## 4 · `pgvector`: store embeddings and run a top-k similarity query

Vector storage powers semantic retrieval (RAG, Ch 13). The `pgvector` extension lets you keep embeddings **in Postgres** — the same database as your rows — and query nearest neighbors with an operator (`<=>` for cosine distance). Here we model the *shape* of that query with a pure-Python cosine over the tiny fixtures in `data/embeddings.json`, so it runs with no extension and no container.

The real Postgres query is, conceptually:

```sql
SELECT id, text
FROM documents
ORDER BY embedding <=> $1      -- cosine distance to the query vector
LIMIT 3;
```

In [ ]:
FX = json.loads((DATA / "embeddings.json").read_text(encoding="utf-8"))

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb) if na and nb else 0.0

def top_k(query_vec, docs, k=3):
    scored = [(cosine(query_vec, d["embedding"]), d) for d in docs]
    scored.sort(key=lambda s: s[0], reverse=True)     # highest cosine similarity first
    return scored[:k]

query = FX["query"]
print(f'query: "{query["text"]}"\n')
print("top-3 most similar documents (cosine):")
for score, doc in top_k(query["embedding"], FX["documents"], k=3):
    print(f"  {score:.3f}  [{doc['id']}]  {doc['text']}")

**What you just saw.** The password/login documents rank highest for a "forgot my password" query — semantic nearest-neighbor search, the retrieval half of RAG. In Postgres this is one indexed query (`ORDER BY embedding <=> $1 LIMIT k`) against an `ivfflat`/`hnsw` index; the *logic* is exactly the cosine ranking above.

### `pgvector` vs a dedicated vector DB — the real trade-off

| | Vectors **in Postgres** (`pgvector`) | **Dedicated** vector DB (Pinecone/Qdrant/Weaviate) |
|---|---|---|
| Backups | One backup covers data **and** vectors | A second system to back up and restore-test |
| Consistency | **Transactional** with the source rows (insert row + embedding atomically) | Two stores to keep in sync; risk of drift |
| Ops surface | One database to provision, secure, monitor | A second store to run forever |
| When it wins | Most systems; "good enough" recall at small/medium scale | **Very large** scale, or you need specialized indexing/filtering |

Keep vectors in Postgres until scale or a specialized indexing/filtering need *clearly* demands the dedicated store. The one-database simplicity is usually worth more than the specialized store's marginal edge.

## 5 · Operating the data layer (§30.6)

Data is the part of the system you **cannot afford to lose**, so operability is mandatory:

- **Backups tested by restoring.** An untested backup is a *hope*, not a recovery plan. Periodically restore into a scratch environment and verify.
- **Read replicas, mind the lag.** Route heavy reads (dashboards, analytics) to **read replicas** to spare the primary — but replicas lag slightly (eventual consistency, Ch 29). Fine for a report; **not** for read-your-writes on a just-saved record.
- **Data lifecycle.** Decide what to **retain**, **archive** to cheap storage, and **delete** — for cost *and* for privacy. The **right to be forgotten** is a concrete data-layer requirement you design for (and it reaches into your caches and backups too).

## 🎯 Senior lens

**Minimize the number of datastores you run.** Every additional store is another thing to provision, secure, back up, monitor, and keep consistent with the others — real, permanent operational cost that never shows up in the demo and always shows up in the on-call rotation. "We'll add a dedicated vector DB / search cluster / document store" should clear a **high** bar: can Postgres — with `pgvector` and full-text search — do this acceptably first? Usually it can, and the one-database simplicity beats the specialized store's marginal edge. The same discipline governs caching: a cache is another store with its own failure mode (staleness), so you add it deliberately, give it a short TTL and explicit invalidation, and never cache what you can't afford to be briefly wrong. Add stores deliberately, not reflexively.

## Recap

- **Cache-aside**: check cache → on miss, fetch + backfill with a TTL. The repeat read is served from memory.
- **LLM response caching** is the same pattern keyed by `(model, inputs)`: identical requests cost **zero tokens** on a hit; Ch 40 extends it to *semantic* (near-identical) matches.
- ⚠️ **Cache invalidation** is the hard part — short TTLs **plus** explicit delete-on-write; ask "how wrong, for how long?"
- **`pgvector`** stores embeddings in Postgres and answers top-k similarity with `ORDER BY embedding <=> $1 LIMIT k` — one backup, transactional consistency.
- Choose a **dedicated vector DB** only at very large scale or for specialized indexing/filtering — on operational cost, not hype.
- **Operate** the data layer: backups **tested by restoring**, read replicas (mind the lag), and a retain/archive/delete **lifecycle** (right-to-be-forgotten).

## 📋 Data-layer readiness checklist

The chapter's closing checklist — run it before shipping a data layer:

- [ ] **Default to Postgres**; every additional datastore justified against its operational cost.
- [ ] **Index** what you filter and join on — and watch the query count for N+1s (30-02).
- [ ] **Pool connections**; add PgBouncer for serverless / many-instance fleets.
- [ ] **Model access patterns first** for any NoSQL store; choose partition keys with care.
- [ ] **Cache** with short TTLs and explicit invalidation on write; never cache what can't be stale.
- [ ] **Keep vectors in `pgvector`** until scale clearly demands a dedicated vector store.
- [ ] **Test backups by restoring them**; route heavy reads to replicas, mindful of lag.
- [ ] **Plan retention, archival, and deletion** for cost and privacy compliance.

## Exercises

Predict before you run.

1. **TTL expiry.** Set the `get_profile` TTL to `ex=1`, read (MISS→backfill), `time.sleep(1.1)`, read again. Predict HIT or MISS the second time, then verify the entry expired.
2. **Cache key sensitivity.** Call `cached_completion` with the same prompt but a *different* `MODEL` string. Predict hit or miss and explain why the model is part of the key.
3. **Hit-rate math.** If 70% of requests are repeats, what's the token-cost reduction from the LLM cache? Compute it from section 3's numbers and state your assumption.
4. **Vector ranking.** Add a 6th document about "password" to `data/embeddings.json` (or in-cell) and rerun `top_k`. Predict whether it displaces the current #3 before you run it.

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your code here


In [ ]:
# Exercise 3 — your code here


In [ ]:
# Exercise 4 — your code here


## Next

- ⬅️ **Previous:** [`30-02-sqlalchemy-pooling-and-the-n1-trap.ipynb`](./30-02-sqlalchemy-pooling-and-the-n1-trap.ipynb).
- 🔧 **The production versions of what you just built:** the cache-aside + LLM cache graduate into [`blueprints/llm/`](../../blueprints/llm/) (and Ch 40's semantic cache); the `pgvector` retrieval path feeds [`blueprints/rag-pipeline/`](../../blueprints/rag-pipeline/). You built the toy; those are the real ones.
- 🏗️ **Capstone:** this establishes `capstone-project/cache/` (Redis cache-aside + LLM cache) and the `pgvector` storage under `capstone-project/rag/` — checkpoint `checkpoints/ch30-data-layer`.
- 🐳 **Live path (`MOCK=0`):** `docker compose up redis postgres` (with the `pgvector` image), set `REDIS_URL` + `ANTHROPIC_API_KEY`; the LLM-cache cell makes ~2 short live calls to seed the cache once.
- See the book **§30.4** (caching, LLM response caching), **§30.5** (search & vector storage), **§30.6** (operating the data layer).